In [2]:
import pandas as pd

ads = pd.read_csv("../data/processed/cleaned_ads.csv")
business = pd.read_csv("../data/raw/business_reports.csv")

In [3]:
business.columns = (
    business.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [4]:
business = business.rename(columns={
    "(parent)_asin": "parent_asin",
    "(child)_asin": "child_asin",
    "sessions_-_total": "sessions_total",
    "unit_session_percentage": "unit_session_pct",
    "ordered_product_sales": "ordered_product_sales",
    "units_ordered": "units_ordered"
})

In [5]:
business[["child_asin", "title", "sessions_total", "unit_session_pct", "ordered_product_sales", "units_ordered"]].head()

,child_asin,title,sessions_total,unit_session_pct,ordered_product_sales,units_ordered
0,B0F4DLPBXG,LUXMIEL Batana Oil for Hair Growth – 100% Pure...,"6,744",7.83%,"$7,274.35",528


In [6]:
ads_summary = {
    "ad_impressions": ads["impressions"].sum(),
    "ad_clicks": ads["clicks"].sum(),
    "ad_spend": ads["total_cost"].sum(),
    "ad_sales": ads["sales"].sum(),
    "ad_purchases": ads["purchases"].sum(),
    "ad_units_sold": ads["units_sold"].sum()
}

ads_summary = pd.DataFrame([ads_summary])
ads_summary

,ad_impressions,ad_clicks,ad_spend,ad_sales,ad_purchases,ad_units_sold
0,694503,5987,5203.77,5497.02,400,408


In [7]:
business_summary = {
    "sessions_total": business["sessions_total"].sum(),
    "units_ordered_total": business["units_ordered"].sum(),
    "ordered_product_sales_total": business["ordered_product_sales"].sum()
}

business_summary = pd.DataFrame([business_summary])
business_summary

,sessions_total,units_ordered_total,ordered_product_sales_total
0,"6,744",528,"$7,274.35"


In [8]:
# Clean business numeric columns
business["sessions_total"] = (
    business["sessions_total"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

business["units_ordered"] = (
    business["units_ordered"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

business["ordered_product_sales"] = (
    business["ordered_product_sales"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

business["unit_session_pct"] = (
    business["unit_session_pct"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

# Convert to numeric
business["sessions_total"] = pd.to_numeric(business["sessions_total"], errors="coerce")
business["units_ordered"] = pd.to_numeric(business["units_ordered"], errors="coerce")
business["ordered_product_sales"] = pd.to_numeric(business["ordered_product_sales"], errors="coerce")
business["unit_session_pct"] = pd.to_numeric(business["unit_session_pct"], errors="coerce")

business[["sessions_total", "units_ordered", "ordered_product_sales", "unit_session_pct"]].dtypes

sessions_total             int64
units_ordered              int64
ordered_product_sales    float64
unit_session_pct         float64
dtype: object

In [9]:
business_summary = {
    "sessions_total": business["sessions_total"].sum(),
    "units_ordered_total": business["units_ordered"].sum(),
    "ordered_product_sales_total": business["ordered_product_sales"].sum()
}

business_summary = pd.DataFrame([business_summary])
business_summary

,sessions_total,units_ordered_total,ordered_product_sales_total
0,6744,528,7274.35


In [10]:
summary = pd.concat([ads_summary, business_summary], axis=1)

summary["ad_ctr"] = summary["ad_clicks"] / summary["ad_impressions"]
summary["ad_cvr"] = summary["ad_purchases"] / summary["ad_clicks"]
summary["acos"] = summary["ad_spend"] / summary["ad_sales"]
summary["roas"] = summary["ad_sales"] / summary["ad_spend"]
summary["session_to_order_rate"] = summary["units_ordered_total"] / summary["sessions_total"]

summary

,ad_impressions,ad_clicks,ad_spend,ad_sales,ad_purchases,ad_units_sold,sessions_total,units_ordered_total,ordered_product_sales_total,ad_ctr,ad_cvr,acos,roas,session_to_order_rate
0,694503,5987,5203.77,5497.02,400,408,6744,528,7274.35,0.008621,0.066811,0.946653,1.056353,0.078292


In [11]:
summary["ad_sales_share"] = summary["ad_sales"] / summary["ordered_product_sales_total"]

In [13]:
keyword_perf = (
    ads.groupby("search_term", dropna=False)
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spend=("total_cost", "sum"),
        sales=("sales", "sum"),
        purchases=("purchases", "sum"),
        units_sold=("units_sold", "sum")
    )
    .reset_index()
)

keyword_perf["ctr"] = keyword_perf["clicks"] / keyword_perf["impressions"]
keyword_perf["cvr"] = keyword_perf["purchases"] / keyword_perf["clicks"]
keyword_perf["acos"] = keyword_perf["spend"] / keyword_perf["sales"]
keyword_perf["roas"] = keyword_perf["sales"] / keyword_perf["spend"]

keyword_perf = keyword_perf.replace([float("inf"), -float("inf")], 0).fillna(0)

keyword_perf.head()

,search_term,impressions,clicks,spend,sales,purchases,units_sold,ctr,cvr,acos,roas
0,0,138,1,0.75,0.0,0,0,0.007246,0.0,0.0,0.0
1,10 & 1 hair growth oil,13,1,0.90,0.0,0,0,0.076923,0.0,0.0,0.0
2,10 in 1 hair growth oil,90,1,0.68,0.0,0,0,0.011111,0.0,0.0,0.0
3,100 0/0 unrefined batana oil,1,1,0.80,0.0,0,0,1.000000,0.0,0.0,0.0
4,100 batana oil,59,1,0.75,0.0,0,0,0.016949,0.0,0.0,0.0


In [14]:
wasted_terms = keyword_perf[(keyword_perf["sales"] == 0) & (keyword_perf["spend"] > 0)]
wasted_terms.head()

,search_term,impressions,clicks,spend,sales,purchases,units_sold,ctr,cvr,acos,roas
0,0,138,1,0.75,0.0,0,0,0.007246,0.0,0.0,0.0
1,10 & 1 hair growth oil,13,1,0.90,0.0,0,0,0.076923,0.0,0.0,0.0
2,10 in 1 hair growth oil,90,1,0.68,0.0,0,0,0.011111,0.0,0.0,0.0
3,100 0/0 unrefined batana oil,1,1,0.80,0.0,0,0,1.000000,0.0,0.0,0.0
4,100 batana oil,59,1,0.75,0.0,0,0,0.016949,0.0,0.0,0.0


In [15]:
campaign_perf = (
    ads.groupby("campaign_name")
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spend=("total_cost", "sum"),
        sales=("sales", "sum"),
        purchases=("purchases", "sum")
    )
    .reset_index()
)

campaign_perf["ctr"] = campaign_perf["clicks"] / campaign_perf["impressions"]
campaign_perf["cvr"] = campaign_perf["purchases"] / campaign_perf["clicks"]
campaign_perf["acos"] = campaign_perf["spend"] / campaign_perf["sales"]
campaign_perf["roas"] = campaign_perf["sales"] / campaign_perf["spend"]

campaign_perf = campaign_perf.replace([float("inf"), -float("inf")], 0).fillna(0)

campaign_perf.head()

,campaign_name,impressions,clicks,spend,sales,purchases,ctr,cvr,acos,roas
0,Auto Campaign 1 Signal Capture,61148,890,631.71,770.46,52,0.014555,0.058427,0.819913,1.219642
1,Auto Campaign 2 Cheap Discovery,1283,67,58.61,98.45,6,0.052221,0.089552,0.595328,1.679747
2,BROAD MATCH CAMPAIGN,219261,1873,1869.09,1818.52,148,0.008542,0.079018,1.027808,0.972944
3,Batana Hair oil Exact Manual,177976,1085,911.91,858.43,58,0.006096,0.053456,1.062300,0.941354
4,Batana Hair oil pharase Manual,54457,721,747.22,798.44,58,0.013240,0.080444,0.935850,1.068547


In [16]:
ads["date"] = pd.to_datetime(ads["date"])

daily_perf = (
    ads.groupby("date")
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spend=("total_cost", "sum"),
        sales=("sales", "sum"),
        purchases=("purchases", "sum")
    )
    .reset_index()
)

daily_perf["ctr"] = daily_perf["clicks"] / daily_perf["impressions"]
daily_perf["cvr"] = daily_perf["purchases"] / daily_perf["clicks"]
daily_perf["acos"] = daily_perf["spend"] / daily_perf["sales"]
daily_perf["roas"] = daily_perf["sales"] / daily_perf["spend"]

daily_perf = daily_perf.replace([float("inf"), -float("inf")], 0).fillna(0)

daily_perf.head()

,date,impressions,clicks,spend,sales,purchases,ctr,cvr,acos,roas
0,2025-06-21,2930,18,16.86,0.00,0,0.006143,0.000000,0.000000,0.000000
1,2025-06-23,166,3,2.10,0.00,0,0.018072,0.000000,0.000000,0.000000
2,2025-06-25,1126,6,3.80,13.99,1,0.005329,0.166667,0.271623,3.681579
3,2025-06-27,2294,9,10.75,0.00,0,0.003923,0.000000,0.000000,0.000000
4,2025-06-28,3040,6,6.46,0.00,0,0.001974,0.000000,0.000000,0.000000


In [17]:
summary.to_csv("../data/processed/executive_summary.csv", index=False)
keyword_perf.to_csv("../data/processed/keyword_performance.csv", index=False)
campaign_perf.to_csv("../data/processed/campaign_performance.csv", index=False)
daily_perf.to_csv("../data/processed/daily_performance.csv", index=False)
wasted_terms.to_csv("../data/processed/wasted_terms.csv", index=False)